# Ask-my-research — RAG over papers + model outputs (Colab, free)

A retrieval-augmented Q&A pipeline over **both** paper text and structured result tables, with **grounded, cited** answers, using a **free open-weight model** (no API key) on Colab's GPU.

**Before running:** Runtime -> Change runtime type -> **T4 GPU** -> Save.

Run the cells top to bottom.


## 1. Install dependencies (~1 min)

In [ ]:
!pip -q install sentence-transformers transformers accelerate torch --upgrade
print('deps installed')

## 2. Write the project files
This recreates the repo structure inside Colab.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data/papers', exist_ok=True)
os.makedirs('data/tables', exist_ok=True)

files = {}

files['src/llm_backend.py'] = r'''#!/usr/bin/env python3
"""
llm_backend.py
==============
A thin, backend-agnostic interface to a chat LLM, so the RAG pipeline runs
against either a self-hosted/open-weight model (via Ollama) or a frontier
API-based model (Anthropic / OpenAI), switchable by one config flag.

This mirrors a real deployment concern: the retrieval and prompt-construction
logic should not care which model generates the final answer.

Backends
--------
- "ollama"    : local, self-hosted/open-weight (default). Free, offline.
                Requires Ollama running locally (https://ollama.com) and a
                pulled model, e.g.:  ollama pull llama3.1:8b
- "anthropic" : Claude via API. Requires ANTHROPIC_API_KEY in the environment.
- "openai"    : GPT via API. Requires OPENAI_API_KEY in the environment.

Usage
-----
    from llm_backend import LLM
    llm = LLM(backend="ollama", model="llama3.1:8b")
    print(llm.complete("Say hello in one word."))
"""

from __future__ import annotations
import os
from typing import Optional


class LLM:
    def __init__(
        self,
        backend: str = "ollama",
        model: Optional[str] = None,
        temperature: float = 0.0,
    ):
        self.backend = backend.lower()
        self.temperature = temperature
        # sensible per-backend default models
        self.model = model or {
            "ollama": "llama3.1:8b",
            "anthropic": "claude-3-5-sonnet-20241022",
            "openai": "gpt-4o-mini",
        }.get(self.backend, "llama3.1:8b")

        if self.backend not in {"ollama", "anthropic", "openai", "transformers"}:
            raise ValueError(f"Unknown backend: {self.backend}")
        if self.backend == "transformers":
            # default to a small, ungated instruct model that runs on Colab's
            # free GPU with no API key and no access request.
            self.model = model or "Qwen/Qwen2.5-1.5B-Instruct"
            self._pipe = None  # lazy-loaded on first call

    # --- public API -------------------------------------------------------
    def complete(self, prompt: str, system: Optional[str] = None) -> str:
        """Return a single completion string for a prompt."""
        if self.backend == "ollama":
            return self._ollama(prompt, system)
        if self.backend == "anthropic":
            return self._anthropic(prompt, system)
        if self.backend == "openai":
            return self._openai(prompt, system)
        if self.backend == "transformers":
            return self._transformers(prompt, system)
        raise RuntimeError("unreachable")

    # --- backends ---------------------------------------------------------
    def _ollama(self, prompt: str, system: Optional[str]) -> str:
        # Local, self-hosted/open-weight. Uses the official `ollama` client if
        # present, else falls back to the REST endpoint via `requests`.
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        try:
            import ollama  # type: ignore
            resp = ollama.chat(
                model=self.model,
                messages=messages,
                options={"temperature": self.temperature},
            )
            return resp["message"]["content"]
        except ImportError:
            import requests
            r = requests.post(
                "http://localhost:11434/api/chat",
                json={
                    "model": self.model,
                    "messages": messages,
                    "stream": False,
                    "options": {"temperature": self.temperature},
                },
                timeout=120,
            )
            r.raise_for_status()
            return r.json()["message"]["content"]

    def _anthropic(self, prompt: str, system: Optional[str]) -> str:
        import anthropic  # pip install anthropic
        client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        resp = client.messages.create(
            model=self.model,
            max_tokens=1024,
            temperature=self.temperature,
            system=system or "",
            messages=[{"role": "user", "content": prompt}],
        )
        # concatenate text blocks
        return "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")

    def _openai(self, prompt: str, system: Optional[str]) -> str:
        from openai import OpenAI  # pip install openai
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        resp = client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
        )
        return resp.choices[0].message.content

    def _transformers(self, prompt: str, system: Optional[str]) -> str:
        # Free, open-weight, no API key. Runs in-process via Hugging Face
        # transformers — ideal for Google Colab's free GPU. The model is
        # lazy-loaded once and reused.
        if self._pipe is None:
            import torch
            from transformers import pipeline
            self._pipe = pipeline(
                "text-generation",
                model=self.model,
                torch_dtype=torch.float16,
                device_map="auto",
            )
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        out = self._pipe(
            messages,
            max_new_tokens=512,
            do_sample=self.temperature > 0,
            temperature=max(self.temperature, 0.01),
            return_full_text=False,
        )
        # transformers returns the generated assistant turn
        gen = out[0]["generated_text"]
        if isinstance(gen, list):  # chat format returns list of messages
            return gen[-1]["content"]
        return gen


if __name__ == "__main__":
    # quick smoke test (requires a running backend)
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--backend", default="ollama")
    ap.add_argument("--model", default=None)
    args = ap.parse_args()
    llm = LLM(backend=args.backend, model=args.model)
    print(llm.complete("Reply with exactly one word: ready"))
'''

files['src/rag.py'] = r'''#!/usr/bin/env python3
"""
rag.py
======
A retrieval-augmented generation pipeline over heterogeneous biomedical
evidence: unstructured paper text AND structured model-output tables
(gene rankings, convergence results). Answers are grounded in retrieved
chunks and returned with explicit source citations.

Design
------
1. INGEST    : load .txt/.md papers and .csv/.tsv result tables; turn each into
               text "chunks", each tagged with a source id and a type
               ("paper" | "table_row").
2. EMBED     : sentence-transformers embeddings (all-MiniLM-L6-v2 by default;
               small, fast, runs on CPU).
3. INDEX     : an in-memory vector store (cosine similarity via numpy). No
               external DB needed; swap in FAISS/Chroma for scale.
4. RETRIEVE  : top-k chunks for a query.
5. GENERATE  : build a grounded prompt that instructs the LLM to answer ONLY
               from the retrieved context and to cite the chunk ids it used;
               refuse if the context is insufficient.

Why heterogeneous retrieval: a question like "which prioritised genes are
ageing-causal and what is the evidence?" is best answered by combining the
model's *structured* output (which genes, what scores) with the *paper text*
(why, with what caveats). Mixing both in one index lets one query draw on both.

Honest scope: this is a research prototype. Retrieval is dense-only (no
re-ranking), the store is in-memory, and groundedness is enforced by prompt
plus citation-checking, not by a separate verifier model.
"""

from __future__ import annotations
import os, glob, re, json
from dataclasses import dataclass, field
from typing import Optional
import numpy as np


# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------
@dataclass
class Chunk:
    id: str            # e.g. "paper:mr_gat#3" or "table:convergent#CTSF"
    text: str
    source: str        # human-readable source name
    kind: str          # "paper" | "table_row"
    meta: dict = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Ingestion
# ---------------------------------------------------------------------------
def _chunk_paragraphs(text: str, target_chars: int = 800):
    """Split on blank lines, then greedily pack paragraphs to ~target_chars."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    out, buf = [], ""
    for p in paras:
        if len(buf) + len(p) + 1 <= target_chars:
            buf = (buf + "\n" + p).strip()
        else:
            if buf:
                out.append(buf)
            buf = p
    if buf:
        out.append(buf)
    return out


def load_papers(papers_dir: str) -> list[Chunk]:
    """Load .txt/.md files as paper chunks."""
    chunks = []
    for path in sorted(glob.glob(os.path.join(papers_dir, "*"))):
        if not path.lower().endswith((".txt", ".md")):
            continue
        name = os.path.splitext(os.path.basename(path))[0]
        with open(path, encoding="utf-8") as f:
            text = f.read()
        for i, ch in enumerate(_chunk_paragraphs(text)):
            chunks.append(Chunk(
                id=f"paper:{name}#{i}",
                text=ch,
                source=f"{name} (paper)",
                kind="paper",
                meta={"file": os.path.basename(path), "chunk": i},
            ))
    return chunks


def load_tables(tables_dir: str) -> list[Chunk]:
    """
    Load .csv/.tsv result tables. Each ROW becomes a chunk verbalised as
    "col1: val1; col2: val2; ...", so structured rows are retrievable by the
    same dense index as the paper text. Expects a header row.
    """
    import csv
    chunks = []
    for path in sorted(glob.glob(os.path.join(tables_dir, "*"))):
        if not path.lower().endswith((".csv", ".tsv")):
            continue
        name = os.path.splitext(os.path.basename(path))[0]
        delim = "\t" if path.lower().endswith(".tsv") else ","
        with open(path, encoding="utf-8") as f:
            reader = csv.DictReader(f, delimiter=delim)
            # use the first column as a row key if it looks like a gene/id
            key_col = reader.fieldnames[0] if reader.fieldnames else "row"
            for i, row in enumerate(reader):
                key = (row.get(key_col) or str(i)).strip()
                verbal = "; ".join(f"{k}: {v}" for k, v in row.items() if v not in (None, ""))
                text = f"[{name}] {verbal}"
                chunks.append(Chunk(
                    id=f"table:{name}#{key}",
                    text=text,
                    source=f"{name} (results table)",
                    kind="table_row",
                    meta={"file": os.path.basename(path), "row": i, "key": key},
                ))
    return chunks


# ---------------------------------------------------------------------------
# Embedding + index
# ---------------------------------------------------------------------------
class DenseIndex:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        self.chunks: list[Chunk] = []
        self.embs: Optional[np.ndarray] = None

    def build(self, chunks: list[Chunk]):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        embs = self.model.encode(texts, normalize_embeddings=True,
                                 show_progress_bar=True, batch_size=64)
        self.embs = np.asarray(embs, dtype=np.float32)
        return self

    def search(self, query: str, k: int = 6, kind: Optional[str] = None):
        q = self.model.encode([query], normalize_embeddings=True)[0]
        sims = self.embs @ q  # cosine (vectors are normalized)
        order = np.argsort(-sims)
        hits = []
        for idx in order:
            c = self.chunks[idx]
            if kind and c.kind != kind:
                continue
            hits.append((float(sims[idx]), c))
            if len(hits) >= k:
                break
        return hits

    # simple persistence so you don't re-embed every run
    def save(self, path: str):
        np.savez(path + ".npz", embs=self.embs)
        with open(path + ".json", "w", encoding="utf-8") as f:
            json.dump([c.__dict__ for c in self.chunks], f)

    def load(self, path: str):
        self.embs = np.load(path + ".npz")["embs"]
        with open(path + ".json", encoding="utf-8") as f:
            self.chunks = [Chunk(**d) for d in json.load(f)]
        return self


# ---------------------------------------------------------------------------
# Grounded generation
# ---------------------------------------------------------------------------
GROUNDED_SYSTEM = (
    "You are a careful biomedical research assistant. Answer ONLY using the "
    "provided context passages. Each passage has an id in [brackets]. Cite the "
    "ids you use, in square brackets, after the sentences they support. If the "
    "context does not contain enough information to answer, say so plainly "
    "rather than guessing. Do not introduce facts not present in the context."
)


def build_prompt(question: str, hits) -> str:
    ctx = "\n\n".join(f"[{c.id}] {c.text}" for _, c in hits)
    return (
        f"Context passages:\n{ctx}\n\n"
        f"Question: {question}\n\n"
        "Answer (grounded in and citing the passages above):"
    )


class RAG:
    def __init__(self, index: DenseIndex, llm):
        self.index = index
        self.llm = llm

    def answer(self, question: str, k: int = 6, mix: bool = True):
        """
        Retrieve top-k and generate a grounded, cited answer.
        If mix=True, ensure both a paper chunk and a table row are represented
        when available, so structured + unstructured evidence are combined.
        """
        hits = self.index.search(question, k=k)
        if mix:
            kinds = {c.kind for _, c in hits}
            for need in ("paper", "table_row"):
                if need not in kinds:
                    extra = self.index.search(question, k=2, kind=need)
                    hits += extra[:1]
        prompt = build_prompt(question, hits)
        answer = self.llm.complete(prompt, system=GROUNDED_SYSTEM)
        return {
            "question": question,
            "answer": answer,
            "sources": [{"id": c.id, "source": c.source, "score": round(s, 3)}
                        for s, c in hits],
        }
'''

files['ask.py'] = r'''#!/usr/bin/env python3
"""
ask.py
======
Build the index over papers + result tables, then answer questions with a
grounded, cited RAG pipeline.

Examples
--------
# 1. Offline self-test (no LLM, no embeddings download needed for the logic check)
python ask.py --selftest

# 2. Build index and ask one question (local Ollama by default)
python ask.py --papers data/papers --tables data/tables \
              --backend ollama --model llama3.1:8b \
              --q "Which prioritised genes are both exercise-responsive and ageing-causal, and what is the evidence?"

# 3. Use an API backend instead
ANTHROPIC_API_KEY=... python ask.py --backend anthropic --q "..."

# 4. Interactive
python ask.py --papers data/papers --tables data/tables --interactive
"""

from __future__ import annotations
import argparse, sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "src"))

from rag import load_papers, load_tables, DenseIndex, RAG  # noqa: E402


def selftest():
    """
    Verify ingestion + retrieval logic with NO model and NO network:
    monkeypatch the embedder with a trivial bag-of-words vectoriser so the
    pipeline's plumbing (chunking, table verbalisation, mixed retrieval) is
    testable offline.
    """
    import numpy as np
    from rag import Chunk

    # tiny synthetic corpus
    papers = [Chunk(id="paper:demo#0",
                    text="Cathepsin F (CTSF) was validated as a longevity-associated "
                         "target by cis-MR and colocalisation.",
                    source="demo (paper)", kind="paper")]
    tables = [Chunk(id="table:convergent#CTSF",
                    text="[convergent] gene: CTSF; exercise_responsive: yes; "
                         "ageing_causal: yes; layer: proteomics",
                    source="convergent (results table)", kind="table_row"),
              Chunk(id="table:convergent#FADS1",
                    text="[convergent] gene: FADS1; exercise_responsive: yes; "
                         "ageing_causal: yes; layer: lipid",
                    source="convergent (results table)", kind="table_row")]
    chunks = papers + tables

    # trivial deterministic "embedder": hashed bag-of-words
    class FakeIndex(DenseIndex):
        def __init__(self):
            self.chunks = []
            self.embs = None
            self._vocab = {}
        def _vec(self, text):
            v = np.zeros(64, dtype=np.float32)
            for w in text.lower().split():
                v[hash(w) % 64] += 1.0
            n = np.linalg.norm(v)
            return v / n if n else v
        def build(self, chunks):
            self.chunks = chunks
            self.embs = np.vstack([self._vec(c.text) for c in chunks])
            return self
        def search(self, query, k=6, kind=None):
            q = self._vec(query)
            sims = self.embs @ q
            order = np.argsort(-sims)
            hits = []
            for idx in order:
                c = self.chunks[idx]
                if kind and c.kind != kind:
                    continue
                hits.append((float(sims[idx]), c))
                if len(hits) >= k:
                    break
            return hits

    idx = FakeIndex().build(chunks)
    hits = idx.search("which gene is a longevity target validated by MR", k=3)
    ids = [c.id for _, c in hits]
    assert any("CTSF" in i for i in ids), f"expected CTSF in {ids}"

    # mixed retrieval: ensure both a paper and a table row can be returned
    paper_hits = idx.search("CTSF", k=2, kind="paper")
    table_hits = idx.search("CTSF", k=2, kind="table_row")
    assert paper_hits and table_hits, "mixed retrieval should find both kinds"

    print("OK: ingestion, chunking, table verbalisation, and mixed dense "
          "retrieval all behave as expected (offline, no model).")
    return 0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--papers", default="data/papers")
    ap.add_argument("--tables", default="data/tables")
    ap.add_argument("--backend", default="ollama")
    ap.add_argument("--model", default=None)
    ap.add_argument("--k", type=int, default=6)
    ap.add_argument("--q", default=None, help="single question")
    ap.add_argument("--interactive", action="store_true")
    ap.add_argument("--embed_model", default="all-MiniLM-L6-v2")
    args = ap.parse_args()

    if args.selftest:
        return selftest()

    # build index
    chunks = load_papers(args.papers) + load_tables(args.tables)
    if not chunks:
        print(f"No data found in {args.papers} or {args.tables}. "
              f"Add .txt/.md papers and .csv/.tsv tables.")
        return 1
    print(f"Ingested {len(chunks)} chunks "
          f"({sum(c.kind=='paper' for c in chunks)} paper, "
          f"{sum(c.kind=='table_row' for c in chunks)} table rows). Embedding...")
    index = DenseIndex(args.embed_model).build(chunks)

    from llm_backend import LLM
    llm = LLM(backend=args.backend, model=args.model)
    rag = RAG(index, llm)

    def show(res):
        print("\n" + "=" * 70)
        print("Q:", res["question"])
        print("-" * 70)
        print(res["answer"])
        print("-" * 70)
        print("Sources:")
        for s in res["sources"]:
            print(f"  [{s['id']}]  {s['source']}  (sim={s['score']})")
        print("=" * 70 + "\n")

    if args.q:
        show(rag.answer(args.q, k=args.k))
    if args.interactive:
        print("Interactive RAG. Empty line to quit.")
        while True:
            try:
                q = input("\nQuestion> ").strip()
            except (EOFError, KeyboardInterrupt):
                break
            if not q:
                break
            show(rag.answer(q, k=args.k))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

files['data/papers/EXAMPLE_mr_gat_summary.md'] = r'''# Example paper summary (placeholder — replace with your own paper text)

This is an EXAMPLE file so the pipeline runs out of the box. Replace the files
in data/papers/ with your own preprint text (.txt or .md).

A multi-omic graph model integrated Mendelian-randomisation causal effect
estimates across five molecular layers (proteomics, methylation, glycomics,
bulk and single-cell transcriptomics) over a protein-protein interaction
network. The model prioritised genes on the exercise-ageing axis and recovered
enrichment for independently defined exercise-responsive and ageing-causal gene
sets that was not detectable in the raw per-gene MR signal.

Eight genes lay at the intersection of all three reference sets and were
described as triple-convergent. Cathepsin F was further examined as a
longevity-associated target using cis-Mendelian randomisation with
colocalisation. The convergence is reported as a statistical convergence rather
than a demonstration of mediation; interventional follow-up is identified as
future work.
'''

files['data/tables/EXAMPLE_convergent_genes.csv'] = r'''gene,exercise_responsive,ageing_causal,vpa_mr_anchored,note
CTSD,yes,yes,yes,lysosomal cathepsin
CTSF,yes,yes,yes,lysosomal cathepsin; longevity target (cis-MR + coloc)
FADS1,yes,yes,yes,lipid metabolism
FADS2,yes,yes,yes,lipid metabolism
HEXIM1,yes,yes,yes,transcriptional regulation
IGFBP7,yes,yes,yes,growth-factor signalling
LTBP3,yes,yes,yes,matrix / TGF-beta
RHOC,yes,yes,yes,cytoskeletal signalling
'''

for path, content in files.items():
    with open(path, 'w') as f:
        f.write(content)
    print('wrote', path)


## 3. Offline self-test (no model) — verifies the pipeline plumbing

In [ ]:
!python ask.py --selftest

## 4. Replace the EXAMPLE data with YOUR data (optional but recommended)

Run this cell to upload your own paper text (.txt/.md) and result tables (.csv/.tsv).
Skip it to just try the example data first.

In [ ]:
from google.colab import files
import shutil, os
print('Upload PAPER files (.txt or .md). Cancel the dialog to skip.')
try:
    up = files.upload()
    for name in up:
        if name.lower().endswith(('.txt','.md')):
            shutil.move(name, f'data/papers/{name}')
            print('-> paper:', name)
        elif name.lower().endswith(('.csv','.tsv')):
            shutil.move(name, f'data/tables/{name}')
            print('-> table:', name)
except Exception as e:
    print('skipped upload:', e)


## 5. Ask a question (free open-weight model on the GPU)

First run downloads the embedding model (~90MB) and the LLM (~1.5B params, ~3GB). Subsequent questions are fast.

In [ ]:
!python ask.py --papers data/papers --tables data/tables \
    --backend transformers --model "Qwen/Qwen2.5-1.5B-Instruct" \
    --q "Which genes are both exercise-responsive and ageing-causal, and what is the evidence?"


## 6. Ask your own questions
Edit the question and re-run.

In [ ]:
!python ask.py --papers data/papers --tables data/tables \
    --backend transformers --model "Qwen/Qwen2.5-1.5B-Instruct" \
    --q "What did the model add beyond the raw MR signal?"
